<h1 style="text-align: center">XGBoost Regressor</h1>

This notebook performs training of the XGBoost regressor on the selected dataset and generates evaluation and interpretation plots.

It includes tuning of the `max_depth` and `best_iteration` hyperparameters:

- `max_depth`: Controls the maximum depth of each decision tree. Deeper trees can model more complex patterns but may lead to overfitting.
- `best_iteration`: The optimal number of trees in the ensemble (boosting rounds). It helps stop training at the right point to avoid overfitting.

### Training
Model training is submitted to the server as parallel LSF jobs. Each job processes a batch of experiments


### Metrics clustermap
After training is completed, a clustermap of regression metrics (e.g., R², Spearman correlation) is plotted. This allows comparing model performance across experiments and configurations.

### SHAP value heatmaps
Feature importance is visualized using heatmaps of absolute SHAP values (|SHAP|).

In addition, a SHAP clustermap with logarithmic color scale is plotted. This improves visualization when SHAP values differ by several orders of magnitude.

### Required parameters
Before running the notebook, you must set the required parameters in the block below (highlighted in red). These parameters define the dataset, model settings, and plotting configuration.

# Outputs

All outputs for a given dataset are stored in: `OUTPUTS/XGB_Regressor/DATASET_NAME/`

There are three main parts:
1) `results.tsv` – one summary row per experiment (`intragenic` / `intergenic`)
2) `predictions_for_barcodes/` – per-barcode true values and predictions
3) `plots/` – all saved figures

---

## 1) Summary table: `results.tsv`

This table summarizes **final model performance** and **feature importance (SHAP)** for each experiment.

**Header + example rows:**
| experiment | table_type | fold | chosen_max_depth | chosen_best_iteration | TRAIN_full_R2 | TRAIN_full_Spearman | HOLDOUT_R2 | HOLDOUT_Spearman | SHAP_ATAC | SHAP_ChromHMM_ReprPCWk |
|------------|------------|------|-------------------|------------------------|---------------|---------------------|------------|-------------------|-----------|------------------------|
| 3intron_5ss_noP_Pool1_Rep2_S106 | intragenic | FINAL | 2 | 548 | 0.446157 | 0.657929 | 0.318314 | 0.557857 | 0.016028 | 0.0 |
| 3intron_5ss_noP_Pool1_Rep2_S106 | intergenic | FINAL | 2 | 548 | 0.413072 | 0.561738 | 0.399150 | 0.547675 | 0.012067 | 0.0 |

### Columns:
- `experiment` – experiment ID
- `table_type` – table type: `intragenic` or `intergenic`
- `fold` – technical column (always `FINAL`, no meaning, kept for compatibility with earlier code)

**Chosen hyperparameters (selected during CV on the training pool):**
- `chosen_max_depth` – selected optimal `max_depth`
- `chosen_best_iteration` – selected optimal number of boosting rounds (early stopping)

**Final model metrics:**
- `TRAIN_full_R2`, `TRAIN_full_Spearman` – quality on the full training set
- `HOLDOUT_R2`, `HOLDOUT_Spearman` – quality on the holdout (test) set

**SHAP feature importance:**
- Columns `SHAP_*` store **mean absolute SHAP values** (|SHAP|) per feature (computed on training data).
- `NaN` → feature was absent or filtered out in this specific experiment (e.g., constant / not present).

---

## 2) Per-barcode predictions: `predictions_for_barcodes/`

Directory: ```OUTPUTS/XGB_Regressor/DATASET_NAME/predictions_for_barcodes```

Files (one per experiment and table type): <br>
```EXPNAME_intragenic_predictions_trainfull_and_holdout.tsv```<br>
```EXPNAME_intergenic_predictions_trainfull_and_holdout.tsv```<br>

These tables contain **true target values and model predictions for each barcode**.

**Header + example rows:**
| experiment | barcode | chr | start | end | target_true | target_pred | split | fold |
|------------|---------|-----|-------|-----|-------------|-------------|--------|------|
| 3intron_5ss_noP_Pool2_S96 | TGTAGACCCCTTATGATC | chr14 | 96229139 | 96229141 | 0.000000 | 0.209147 | train_full | FINAL |
| 3intron_5ss_noP_Pool2_S96 | TCATTCATCGATGCGCAA | chr17 | 46707724 | 46707726 | 0.000000 | 1.519373 | train_full | FINAL |

### Columns:
- `experiment` – experiment ID
- `barcode` – barcode sequence
- `chr`, `start`, `end` – genomic coordinates of the barcode
- `target_true` – ground-truth target value
- `target_pred` – model prediction
- `split` – where this barcode belongs:
  - `train_full` – barcode from the training pool (predictions after fitting the final model on full training)
  - `holdout` – barcode from the holdout/test set
- `fold` – technical column (always `FINAL`)

Pipeline logic (short):
- CV is performed on the training pool to pick hyperparameters.
- Then the **final model** is trained on the full training set.
- Predictions are saved for both `train_full` and `holdout`.

---

## 3) Plots

All figures produced by the pipeline are saved in:
```OUTPUTS/XGB_Regressor/DATASET_NAME/plots```

<h1 style="text-align: center">RUN CODE</h1>


In [ ]:
import pandas as pd
import os
import math, os, subprocess, shlex, re, time
import sys
from pathlib import Path
from settings.utils import find_project_root, resolve_path, PROJECT_ROOT
sys.path.append("settings/custom_functions")

## Parameters

This block defines all configurable parameters for the ML training pipeline and LSF job submission.  
Only this section should be edited when running a new dataset or tuning hyperparameters.

### Input paths

- `INPUT_DIR` – directory with the training tables (`*.alldata.tab`).  
- `EXP_LIST` – text file with experiment names (one per line), controls which tables are processed.  
- `ANNOT_DROP_COLS` – file listing annotation columns to remove before modeling.  
- `INTRA_DROP_COLS` – file listing additional columns to drop for intragenic tables.

---

### Feature settings

- `CATEGORICAL_COLUMNS` – comma-separated list of columns treated as categorical (e.g. `ChromHMM`).  
  These columns will be one-hot encoded before model training.

---

### XGBoost hyperparameters

- `DEPTH_GRID` – comma-separated list of `max_depth` values for grid search.  
- `NUM_BOOST_ROUND` – maximum number of boosting iterations.  
- `EARLY_STOP` – early stopping rounds (training stops if no improvement after N rounds).

---

<span style="color:red; font-weight:bold;font-size:20px;">
Edit the paths in the cell below.
</span>

In [ ]:
# Parameters you want to feed into the python script via environment variables
DATASET_NAME = 'single_combined'
EXP_LIST = f"DATA/{DATASET_NAME}/names_all_clean.txt"

# for example "ChromHMM,AnotherColumn,ThirdColumn"
CATEGORICAL_COLUMNS = "ChromHMM,SPIN,GC_category,RepeatGroup,annot_rna_type,Chr"

# LOG_TARGET=1 trains on log10(1 + target). Needed for single_combined, where the
# exp.log10 column holds raw meanRPM (0..40607, 35% zeros). Set to "" to keep the target
# as it is in the table.
LOG_TARGET = 1

# DEPTH_GRID = "3,4,5,6,7"
# NUM_BOOST_ROUND = 15000
# EARLY_STOP = 2000

DEPTH_GRID = "5,6"
NUM_BOOST_ROUND = 3000
EARLY_STOP = 500

# LSF / bsub params (your style)
# LINES_PER_JOB = 3 # Batch size: how many experiments each parallel worker processes
LINES_PER_JOB = 1
QUEUE = "short-gpu"
GPU_SPEC = "num=1:j_exclusive=yes:gmem=17G:gmodel=NVIDIAA40"
MEM_MB = 4000
AFFINITY = "thread*1"
JOB_NAME = 'expJob'
WAIT_FOR_LSF = True # needed for `jupyter execute`: wait until the job array finishes before merging outputs

## Default submission block

The parameters below are set by default and **do not require modification** to run the script.
This cell builds and submits an LSF array job to the GPU queue.

In [ ]:
# For Input
INPUT_DIR = f"tables_for_training/{DATASET_NAME}"
ANNOT_DROP_COLS = "settings/columns_to_drop_in_intergenic.txt"
INTRA_DROP_COLS = "settings/columns_to_drop_in_intragenic.txt"
OUTPUT_DIR_PATH = f"OUTPUTS/XGB_Regressor/{DATASET_NAME}"

# For Output
BARCODE_PREDS_PATH = f'{OUTPUT_DIR_PATH}/predictions_for_barcodes'
LOG_ERR = 'MODELS/XGB_regressor/artifacts/log_err_LSF/err_%I.txt'
LOG_OUT = "MODELS/XGB_regressor/artifacts/log_out_LSF/out_%I.txt"
SCRIPT = "MODELS/XGB_regressor/XGB_basic_DS_regressor.py"

PYTHON_BIN = sys.executable

# Resolve paths
INPUT_DIR_p = resolve_path(INPUT_DIR)
EXP_LIST_p = resolve_path(EXP_LIST)
ANNOT_DROP_COLS_p = resolve_path(ANNOT_DROP_COLS)
INTRA_DROP_COLS_p = resolve_path(INTRA_DROP_COLS)
OUTPUT_DIR_PATH_p = resolve_path(OUTPUT_DIR_PATH)
BARCODE_PREDS_PATH_p = resolve_path(BARCODE_PREDS_PATH)
LOG_ERR_p = resolve_path(LOG_ERR)
LOG_OUT_p = resolve_path(LOG_OUT)
SCRIPT_p = resolve_path(SCRIPT)


# Compute array size
with open(EXP_LIST_p, "r") as f:
    exp_names = [line.strip() for line in f if line.strip()]

n_exp = len(exp_names)
n_jobs = max(1, math.ceil(n_exp / LINES_PER_JOB))

print(f"Experiments: {n_exp}")
print(f"LINES_PER_JOB: {LINES_PER_JOB}")
print(f"Job array size: {n_jobs} (array jobs).")
print("Note: LSF will run as many array jobs concurrently as resources allow (concurrent jobs).")


# Build command
os.makedirs(OUTPUT_DIR_PATH_p, exist_ok=True)
os.makedirs(BARCODE_PREDS_PATH_p, exist_ok=True)
os.makedirs(LOG_OUT_p.parent, exist_ok=True)
os.makedirs(LOG_ERR_p.parent, exist_ok=True)
os.makedirs(OUTPUT_DIR_PATH_p / "plots", exist_ok=True)

# deleting old log files
!rm -f {str(resolve_path("MODELS/XGB_regressor/artifacts/log_err_LSF/*.txt"))}
!rm -f {str(resolve_path("MODELS/XGB_regressor/artifacts/log_out_LSF/*.txt"))}
!rm -f {str(resolve_path("MODELS/XGB_regressor/artifacts/log_training/*.log"))}
!rm -f {str(resolve_path("MODELS/XGB_regressor/artifacts/log_training_errors/*.err"))}

# Export env vars for the script (script must read os.getenv / os.environ)
env = (
    f'EXP_LIST="{str(EXP_LIST_p)}" '
    f'LINES_PER_JOB="{LINES_PER_JOB}" '
    f'INPUT_DIR="{str(INPUT_DIR_p)}" '
    f'OUTPUT_DIR_PATH="{str(OUTPUT_DIR_PATH_p)}" '
    f'BARCODE_PREDS_PATH="{str(BARCODE_PREDS_PATH_p)}" '
    f'ANNOT_DROP_COLS="{str(ANNOT_DROP_COLS_p)}" '
    f'INTRA_DROP_COLS="{str(INTRA_DROP_COLS_p)}" '
    f'DEPTH_GRID="{DEPTH_GRID}" '
    f'CATEGORICAL_COLUMNS="{CATEGORICAL_COLUMNS}" '
    f'NUM_BOOST_ROUND="{NUM_BOOST_ROUND}" '
    f'EARLY_STOP="{EARLY_STOP}" '
    f'LOG_TARGET="{LOG_TARGET}" '
    f'PROJECT_ROOT="{str(PROJECT_ROOT)}" '
)


cmd = f"""
{env} bsub -J "{JOB_NAME}[1-{n_jobs}]" \
-q {QUEUE} \
-gpu {shlex.quote(GPU_SPEC)} \
-R rusage[mem={MEM_MB}] \
-R affinity[{AFFINITY}] \
-oo "{str(LOG_OUT_p)}" \
-eo "{str(LOG_ERR_p)}" \
{PYTHON_BIN} "{str(SCRIPT_p)}"
""".strip()

# Make it one line for ! execution
cmd_one_line = " ".join(cmd.split())
print("\nSubmitting command:\n", cmd_one_line, "\n")
print()

remove old outputs

In [ ]:
output_splitted_p = resolve_path("MODELS/XGB_regressor/artifacts/output_splitted")
grid_selection_p = resolve_path("MODELS/XGB_regressor/artifacts/grid_selection")
log_err_lsf_p = resolve_path("MODELS/XGB_regressor/artifacts/log_err_LSF")
log_out_lsf_p = resolve_path("MODELS/XGB_regressor/artifacts/log_out_LSF")
log_training_p = resolve_path("MODELS/XGB_regressor/artifacts/log_training")
log_training_errors_p = resolve_path("MODELS/XGB_regressor/artifacts/log_training_errors")

! rm -rf {output_splitted_p}
! rm -rf {grid_selection_p}
! rm -rf {log_err_lsf_p}
! rm -rf {log_out_lsf_p}
! rm -rf {log_training_p}
! rm -rf {log_training_errors_p}

! rm -rf {OUTPUT_DIR_PATH_p / "plots"}
! rm -rf {BARCODE_PREDS_PATH_p}
! rm -f {OUTPUT_DIR_PATH_p / "results.tsv"}

# Recreate directories after cleanup so a fresh `jupyter execute` run has stable paths.
os.makedirs(output_splitted_p, exist_ok=True)
os.makedirs(grid_selection_p, exist_ok=True)
os.makedirs(log_err_lsf_p, exist_ok=True)
os.makedirs(log_out_lsf_p, exist_ok=True)
os.makedirs(log_training_p, exist_ok=True)
os.makedirs(log_training_errors_p, exist_ok=True)
os.makedirs(OUTPUT_DIR_PATH_p, exist_ok=True)
os.makedirs(BARCODE_PREDS_PATH_p, exist_ok=True)
os.makedirs(OUTPUT_DIR_PATH_p / "plots", exist_ok=True)

<span style="color:red; font-weight:bold;font-size:20px;">
Submit to the queue
</span>

In [ ]:
submit = subprocess.run(cmd_one_line, shell=True, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print(submit.stdout)
if submit.returncode != 0:
    raise RuntimeError(f"bsub failed: {submit.returncode}")

match = re.search(r"Job <(\d+)>", submit.stdout)
job_id = match.group(1) if match else None

if WAIT_FOR_LSF:
    if not job_id:
        raise RuntimeError("Cannot parse LSF job id")

    print(f"Waiting for LSF job {job_id}...")

    while True:
        out = subprocess.run(f"bjobs -noheader -o stat {job_id}",
                             shell=True, text=True,
                             stdout=subprocess.PIPE).stdout

        statuses = [s.split()[0] for s in out.splitlines() if s.strip()]
        if not any(s in {"PEND", "RUN", "SUSP"} for s in statuses):
            break

        print(f"Statuses: {', '.join(statuses)}")
        time.sleep(30)

    final = subprocess.run(f"bjobs -a -noheader -o stat {job_id}",
                           shell=True, text=True,
                           stdout=subprocess.PIPE).stdout

    final_statuses = [s.split()[0] for s in final.splitlines() if s.strip()]
    print(f"Final: {', '.join(final_statuses)}")

    if "EXIT" in final_statuses:
        raise RuntimeError(f"LSF job {job_id} failed")

---
You can check whether the computations have finished and the job has completed successfully:

In [ ]:
!bjobs

Python script errors are saved in ```MODELS/XGB_regressor/artifacts/log_training_errors```, and LSF errors are saved in ```MODELS/XGB_regressor/artifacts/log_err_LSF```. We will now check all log files from each parallel job

#### Let's check if there's anything in the error log files (they will appear when the calculations begin). They should be empty. If not, the calculation failed.

In [ ]:
dirs = [
    "MODELS/XGB_regressor/artifacts/log_training_errors",
    "MODELS/XGB_regressor/artifacts/log_err_LSF"
]

for d in dirs:
    folder_name = os.path.basename(d)
    print(f"\n=== Checking: {folder_name} ===")
    if not os.path.isdir(d):
        print("Directory not found.")
        continue
    files_found = False
    for fname in sorted(os.listdir(d)):
        path = os.path.join(d, fname)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            files_found = True
            print("\n" + "="*80)
            print(f"FILE: {fname}\n")
            with open(path, "r", errors="replace") as f:
                print(f.read())
    if not files_found:
        print("No non-empty files.")

---

#### Merge split results

During parallel execution, each array job produces a separate `.tsv` result file.  
In this cell, all generated result files are merged into a single final table.

You can modify the `output_file_path_final` variable to change where the merged table will be saved.

In [ ]:
# where the split result files are stored
output_dir_path = 'MODELS/XGB_regressor/artifacts/output_splitted'
output_dir_path_p = resolve_path(output_dir_path)

# where to save the final merged file (this is a FILE, not a directory)
output_file_path_final = OUTPUT_DIR_PATH_p / "results.tsv"

# --- preparation ---
output_file_path_final.parent.mkdir(parents=True, exist_ok=True)

# take only .tsv files
files = sorted([
    p for p in output_dir_path_p.iterdir()
    if p.is_file()
    and not p.name.startswith(".")
    and p.suffix == ".tsv"
])

if not files:
    raise FileNotFoundError(f"No .tsv files found in: {output_dir_path_p}")

# --- merging ---
dfs = []
for fp in files:
    df_part = pd.read_csv(fp, sep="\t")
    dfs.append(df_part)

DATA = pd.concat(dfs, axis=0, ignore_index=True)

# === Sanity check: number of processed experiments ===
expected_n = 2 * len(exp_names) # two tables: intra- and intergenic per experiment
actual_n = len(DATA)

if actual_n != expected_n:
    print(
        f"[WARNING] Experiment count mismatch: results.tsv rows={actual_n}, expected experiments={expected_n}."
        "It seems that not all experiments were processed "
        "or some tables failed to load/compute."
    )
else:
    print(f"[OK] All experiments processed correctly: n = {actual_n}")


DATA.to_csv(output_file_path_final, sep="\t", index=False)
print(f"[OK] Merged {len(files)} files -> {output_file_path_final} | rows={len(DATA)}")

### Viewing the final table `results.tsv`:

In [ ]:
DATA = pd.read_csv(output_file_path_final, sep="\t")
DATA

## Metric clustermaps (Train / Holdout)

In this cell, we generate clustermaps of model performance metrics  
(R², Spearman) for both training and holdout datasets.

The plotting is performed using a custom function located in:

`settings/custom_functions/`

The complete list of parameters is in the docstring of the function.

---

### Key parameters


- `table_type` (**most important**)  
  Defines which tables are visualized:  
  - `"intragenic"`  
  - `"intergenic"`
  
- `dataset_name`  
  Used only for figure titles and labeling.  
  Does not affect computations.
  
- `save`  
  If `True`, the figure is saved to disk.

- `outdir`  
  Directory where plots will be stored.

---

### Figure size behavior

The function is implemented with automatic figure size and font scaling.  
However, in some cases auto-sizing may behave suboptimally.

If needed, you can manually control:

- `figsize` – fixed figure size (tuple)  
- `title_font_size` – explicit title font size
- `axis_font_size`

Some technical logs may be displayed during execution. You can safely ignore them, they are printed for debugging purposes only.

In [ ]:
from print_plots_regression import prepare_agg_df_for_plots, plot_metric_clustermap

agg_df = prepare_agg_df_for_plots(DATA)
plots_dir = OUTPUT_DIR_PATH_p / "plots"

plot_metric_clustermap(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intragenic",
    model="XGBoost",
    split="both",

    width_per_exp=5,
    height=7,

    title_font_size=32,
    axis_font_size=20,
    cell_font_size=18,

    save=True,
    outdir=plots_dir,
)

plot_metric_clustermap(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intergenic",
    model="XGBoost",
    split="both",

    width_per_exp=5,
    height=7,

    title_font_size=32,
    axis_font_size=20,
    cell_font_size=18,

    save=True,
    outdir=plots_dir,
)


## SHAP clustermap (absolute values)

Next, we plot a clustermap of **absolute SHAP values** (|SHAP|) for the model trained separately for each experiment.

Features that were **absent during training** (missing in a given dataset) or were **dropped** (e.g., constant columns or filtered during preprocessing) are shown as **black cells** in the heatmap.

### `white_pct` parameter

`white_pct` controls how many of the smallest |SHAP| values are treated as effectively zero and therefore colored white.

It defines a lower-tail cutoff (quantile-like threshold): the bottom `white_pct` fraction of values is mapped to zero for visualization.  
This suppresses very small effects and makes the color scale more informative.


During execution, the function prints several technical values to the logs:

- `white_pct` – the chosen lower-tail fraction  
- `ε (epsilon)` – the computed threshold below which |SHAP| values are set to zero  
- `Vmax` – the maximum value used for colormap scaling  

Feature groups are shown in distinct colors. The grouping of features is defined by the get_groups_of_features() function, and the color mapping for each group is specified in the group_colors dictionary shown below.

In [ ]:
from print_plots_regression import plot_shap_clustermap_pct

# Feature groups are defined here instead of get_groups_of_features(): that function lists
# the signal tracks of set1_combined_1, none of which exist in this dataset. The one-hot
# families are collected by prefix.
shap_feats = [c.replace("SHAP_", "") for c in agg_df.columns if c.startswith("SHAP_")]


def by_prefix(prefix):
    return [f for f in shap_feats if f.startswith(prefix)]


groups_to_features = {
    "annotations": ["MANE_all", "MANE_exonic", "MANE_antisense", "gencode_noncoding",
                    "MANElength", "DistToTSS", "DistToTES", "RelPos",
                    "DistToIntron5", "DistToIntron3", "RelIntronPos"] + by_prefix("annot_rna_type_"),
    "mane_expression": ["MANE_all_exp.norm.log10"],
    "chromhmm":        by_prefix("ChromHMM_"),
    "nuclear_org":     by_prefix("SPIN_"),
    "gc_content":      ["GC"] + by_prefix("GC_category_"),
    "repeats":         by_prefix("RepeatGroup_"),
    "chromosome":      by_prefix("Chr_"),
    "transcriptomics": ["closestCAGEdist"],
    "global_norm":     ["meanNormRPM", "meanMapRPM"],
    "nmd":             ["NMD_expected"],
}

group_colors = {
    "annotations": "tab:brown",
    "mane_expression": "tab:olive",
    "chromhmm": "tab:cyan",
    "nuclear_org": "tab:purple",
    "gc_content": "tab:pink",
    "repeats": "tab:red",
    "chromosome": "tab:gray",
    "transcriptomics": "tab:orange",
    "global_norm": "tab:blue",
    "nmd": "tab:green",
}

# Anything left ungrouped stays black on the heatmap, so print it rather than lose it
ungrouped = set(shap_feats) - {f for flist in groups_to_features.values() for f in flist}
print(f"[INFO] features: {len(shap_feats)} | ungrouped: {sorted(ungrouped)}")

plot_shap_clustermap_pct(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intragenic",
    model="XGBoost",

    width=40,
    height_per_exp=0.63,

    title_font_size=50,
    x_tick_font_size=12,
    y_tick_font_size=12,

    white_pct=20.0,
    dropnan=True,

    save=True,
    outdir=plots_dir,

    groups_to_features=groups_to_features,
    group_colors=group_colors,
)

plot_shap_clustermap_pct(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intergenic",
    model="XGBoost",

    width=40,
    height_per_exp=0.63,

    title_font_size=50,
    x_tick_font_size=12,
    y_tick_font_size=12,

    white_pct=20.0,
    dropnan=True,

    save=True,
    outdir=plots_dir,

    groups_to_features=groups_to_features,
    group_colors=group_colors,
)

### Log-scale SHAP clustermap

This function plots a clustermap of SHAP values using logarithmic color normalization (`LogNorm`). This allows better visualization when SHAP values span several orders of magnitude.

The parameter `white_pct` defines a lower percentile threshold for SHAP values, similar to the function `plot_shap_clustermap_pct`. The corresponding SHAP value at this percentile is used as the minimum color scale threshold (`vmin`). All SHAP values below this threshold are displayed in white. Additionally, zero and missing values (NaN) are masked and also shown in white.

Thus, low-magnitude SHAP values are visually suppressed, improving contrast and interpretability of more important features. All other preprocessing and visualization steps are the same as in `plot_shap_clustermap_pct`, including hierarchical clustering and feature sorting by mean absolute SHAP value.

In [ ]:
from print_plots_regression import plot_shap_clustermap_log_pct

plot_shap_clustermap_log_pct(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intragenic",
    model="XGBoost",

    width=40,
    height_per_exp=0.63,

    title_font_size=32,
    x_tick_font_size=12,
    y_tick_font_size=12,

    white_pct=20.0,
    dropnan=True,

    save=True,
    outdir=plots_dir,

    groups_to_features=groups_to_features,
    group_colors=group_colors,
)

plot_shap_clustermap_log_pct(
    agg_df=agg_df,
    dataset_name=DATASET_NAME,
    table_type="intergenic",
    model="XGBoost",

    width=40,
    height_per_exp=0.63,

    title_font_size=32,
    x_tick_font_size=12,
    y_tick_font_size=12,

    white_pct=20.0,
    dropnan=True,

    save=True,
    outdir=plots_dir,

    groups_to_features=groups_to_features,
    group_colors=group_colors,
)